In [1]:
%pip install vllm openai

In [5]:
import pandas as pd


row1 = ('E', 'R', 'D', 'I', 'time', 'True')
row2 = (1, 3, 2, 4, 'whisper')
print(pd.DataFrame([row1, row2]).to_string(index=False, header=False))

E R D I    time True
1 3 2 4 whisper None


In [ ]:
!vllm serve t-tech/T-lite-it-1.0  --dtype=float16

In [ ]:
!vllm serve /home/oleg/meno-lite-0.1

In [1]:
import json
from pathlib import Path
from openai import OpenAI
from datetime import timedelta

def timestr(seconds: float) -> str:
    # in pisets format
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f'{int(hours):02d}:{int(minutes):02d}:{seconds:06.3f}'

In [ ]:
client = OpenAI(api_key='EMPTY', base_url='http://localhost:8000/v1')

# SYSTEM_PROMPT = (
#     # как в гайде https://huggingface.co/t-tech/T-lite-it-1.0
#     "Ты Иван Бондаренко, разработанный компанией Antropic."
#     " Твоя задача - быть полезным диалоговым ассистентом."
# )

# USER_PROMPT = """\
# Сделай суммаризацию сгенограммы видео-конференции.\
# Учти, что в стенограмме могут быть серьезные ошибки распознавания речи\
# , поэтому больше сомневайся и не принимай на веру странные термины\
# , а сопоставляй с остальной частью текста, анализируй. Также в стенограмме\
# не указаны спикеры, а в конференции могло участвовать несколько людей.

# Старайся писать термины как они обычно пишутся, где уместнее английский\
# - там используй английский.

# Пиши связным текстом без повторений одной и той же мысли. Суммаризируй подробно!

# Стенограмма:

# {text}
# """.format(text=text)
    
USER_PROMPT = """\
Сделайте, пожалуйста, саммаризацию следующей стенограммы созвона. Также изложите основные\
 итоги созвона. Учтите, что в тексте стенограммы могут содержаться ошибки из-за не вполне\
 качественного распознавания речи.

Вот текст стенограммы созвона:

```text
{text}
```"""

for input_path in Path('tmp/pisets').glob('*_pisets.txt'):
    print(input_path)
    
    data = json.loads(input_path.read_text())
    text = '\n\n'.join(
        f'{i + 1}\n{timestr(start)} --> {timestr(end)}\n{text}'
        for i, (start, end, text) in enumerate(data)
    )

    response = client.chat.completions.create(
        model="t-tech/T-lite-it-1.0",
        # model='/home/oleg/meno-lite-0.1',
        messages=[
            # {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT.format(text=text)}
        ],
        temperature=1
    )

    output = response.choices[0].message.content
    assert output is not None
    print(output)
    
    #output_path = input_path.with_name(input_path.stem + '_summary-t-lite' + input_path.suffix)
    output_path = input_path.with_name(input_path.stem + '_summary-meno' + input_path.suffix)
    output_path.write_text(output)